# Notebook 1 — Train Sen2-SSIR

This notebook is the complete interface for data preparation and training. Place one or more Sentinel-2 L2A `.SAFE`, metadata XML, or GDAL-readable ZIP products under `Data/Training_Scene` before running it.

In [ ]:
import tensorflow as tf

from utils import (
    PREPARED_DATA_DIR, PRETRAINED_WEIGHTS, TRAINING_RESULTS_DIR,
    TRAINING_SCENE_DIR, PatchConfig, TrainingConfig,
    discover_scenes, ensure_project_directories,
    prepare_training_scenes, train_sen2_ssir,
)
from utils.visualization import plot_training_history

ensure_project_directories()
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Pretrained weights:', PRETRAINED_WEIGHTS)

## Configuration

Choose `scratch` to train from random initialization, `pretrained` to fine-tune the published checkpoint, or `resume` to continue `Results/Training/Sen2-SSIR_latest.h5` with optimizer state. `epochs` is the total target epoch count, including epochs already completed by a resumed run.

In [ ]:
MODE = 'pretrained'       # 'scratch', 'pretrained', or 'resume'
OVERWRITE_PREPARED = False

patch_config = PatchConfig(
    training_patch_size=32,
    patches_per_scene=8000,
    inference_patch_size=128,
    inference_border=4,
    seed=42,
)
training_config = TrainingConfig(
    learning_rate=1e-4,
    batch_size=128,
    epochs=1500,
    validation_fraction=0.10,
    early_stopping_patience=50,
    reduce_lr_patience=15,
    reduce_lr_cooldown=20,
    minimum_learning_rate=1e-5,
    seed=42,
)

## Prepare the training scenes

Gaussian degradation, 2× mean aggregation, bilinear interpolation, and 8,000 random aligned patches per scene. Existing complete prepared scenes are reused unless `OVERWRITE_PREPARED` is true.

In [ ]:
training_scenes = discover_scenes(TRAINING_SCENE_DIR)
if not training_scenes:
    raise RuntimeError(f'Place at least one Sentinel-2 L2A product in {TRAINING_SCENE_DIR}')
print(f'Found {len(training_scenes)} training scene(s):')
for scene in training_scenes:
    print(' -', scene.name)

prepared_scenes = prepare_training_scenes(
    TRAINING_SCENE_DIR,
    PREPARED_DATA_DIR,
    patch_config=patch_config,
    overwrite=OVERWRITE_PREPARED,
)
print(f'Prepared {len(prepared_scenes)} scene dataset(s) in {PREPARED_DATA_DIR}')

## Train Sen2-SSIR

The latest full model is saved after every epoch for reliable resume. Best and final weights, a CSV history, and the run configuration are also retained.

In [ ]:
model, history = train_sen2_ssir(
    PREPARED_DATA_DIR,
    TRAINING_RESULTS_DIR,
    mode=MODE,
    config=training_config,
    pretrained_weights=PRETRAINED_WEIGHTS,
)
print('Training artifacts:', TRAINING_RESULTS_DIR)

In [ ]:
plot_training_history(TRAINING_RESULTS_DIR / 'training_history.csv')